# Avance 3 · Baseline de regresión de la potencia (fase 2)

**Bioactive AI** · Equipo 7 · Maestría en Inteligencia Artificial Aplicada (TC5035.10)

Este cuaderno es el compañero del [03_Baseline](03_Baseline.ipynb) para la fase 2. El baseline del Avance 3 clasificaba el par compuesto-diana en activo o inactivo a 10 µM; después la variable objetivo cambió: lo que se predice es la **potencia** (`pActividad`) y la clase se deriva de ella al final ([2.9](../docs/decisiones.md#d2-9)). Así que el baseline también es una regresión, y esto es lo que lo establece.

Las preguntas de la rúbrica del Avance 3, aplicadas aquí:

- **¿Qué algoritmo?** Tres, que se distinguen por lo que ven: el promedio (el piso), una recta con las mediciones exactas y **la misma recta con las censuradas como cotas** (regresión de Tobit).
- **¿Qué características pesan?** Los coeficientes de la recta, con y sin censura, y si coinciden.
- **¿Sobreajusta?** El error en entrenamiento contra el de validación.
- **¿Qué métrica?** La concordancia de Harrell, que usa también las censuradas; Spearman y RMSE solo con las exactas, declarado así.
- **¿Qué desempeño mínimo?** No hay un histórico. El piso es el promedio; el techo, el ruido propio de la medición.


## 1. Preparación

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"          # un solo hilo: la columna de tiempos es comparable

import platform
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import scipy

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
PROCESADOS = RAIZ / "data" / "processed"

from src.evaluation.regresion import RUIDO_EXPERIMENTAL_IC50, RUIDO_EXPERIMENTAL_KI
from src.models.baseline_regresion import (ALFA_RIDGE, baselines, evaluar_baselines,
                                           intervalo_util, tabla)

# Versiones con las que se ejecutó: tiene que decir Python 3.12.
print(f"sistema  {platform.platform()}")
print(f"python   {platform.python_version()}")
for m in (np, pd, sklearn, scipy):
    print(f"{m.__name__:<8} {m.__version__}")

### Los datos

La unidad es el par compuesto-diana, con los mismos pares de entrenamiento y validación que usa el [04](04_Models.ipynb) (la prueba reservada no se toca) y la diana como columnas de 0 y 1. Lo que cambia respecto de la clasificación es qué filas entran: **ninguna se descarta por no decidir a 10 µM**, porque una cota como `> 1 µM` no decide a 10 µM pero sí informa a un modelo con censura. Salen solo las filas que no informan nada: las que no tienen ninguna cota y las de cotas que se contradicen (intervalo vacío).

In [ ]:
pares = pd.read_parquet(PROCESADOS / "fase2_pares.parquet")
desc = pd.read_parquet(PROCESADOS / "fase2_descriptores_limpios.parquet")
d = pares[pares["particion"].ne("prueba")].merge(desc, on="smiles").reset_index(drop=True)
cols = [c for c in desc.columns if c != "smiles"]
dianas = pd.get_dummies(d["target_gene"])

inferior = d["limite_inferior"].to_numpy(float)
superior = d["limite_superior"].to_numpy(float)
util = intervalo_util(inferior, superior)
print(f"{len(d):,} pares de entrenamiento y validación · {(~util).sum():,} sin información "
      f"(sin cota o con cotas cruzadas) y salen")

d, dianas = d[util].reset_index(drop=True), dianas[util].reset_index(drop=True)
inferior, superior = inferior[util], superior[util]
X = np.hstack([d[cols].to_numpy("float32"), dianas.to_numpy("float32")])
nombres = cols + list(dianas.columns)
exacta = inferior == superior
clase = d["activo_10um"].to_numpy(float)          # la clase derivada a 10 µM; NaN si no decide
grupos = d["grupo_cv"].to_numpy()
procedencia = d["estrato"].to_numpy()             # solo_chembl, solo_npass, mixto
print(f"{len(d):,} pares · {exacta.sum():,} con potencia exacta ({exacta.mean():.1%}) · "
      f"{(~exacta).sum():,} solo con cotas · {np.isnan(clase).sum():,} no deciden a 10 µM")

## 2. Tres baselines

| Baseline | Qué ve | Para qué |
|:--|:--|:--|
| `media` | las exactas; predice siempre su promedio | el piso: un modelo que no le gane no aprendió nada |
| `ridge_exactas` | solo las exactas | la regresión lineal que un baseline sencillo haría |
| `tobit` | las exactas **y** las censuradas como cotas | la misma recta que Ridge, con la censura |

`ridge_exactas` y `tobit` son la misma familia con la misma penalización (`alfa = 100` de Ridge, escrita en la escala de cada uno), así que si difieren es por lo que cada uno ve y no por el modelo. El Tobit supone que la potencia es normal alrededor de la recta y maximiza la verosimilitud de lo que cada fila sí dice: una exacta aporta su valor y un `> 10 µM` la probabilidad de caer por debajo de pActividad 5 ([`src/models/tobit.py`](../src/models/tobit.py)).

La validación es la del Avance 3: cinco pliegues agrupados por `grupo_cv` (andamio y esqueleto de conectividad), semillas 0, 1 y 2. Las métricas de regresión usan el intervalo completo; MCC y AUROC se calculan sobre la clase derivada, cortando la potencia predicha en 5 (= 10 µM), para poder compararlas con el baseline de clasificación. Un estrato se reporta solo si tiene 20 casos de cada clase.

In [ ]:
res = evaluar_baselines(X, inferior, superior, grupos, procedencia, clase)
print(f"{len(res):,} filas: {res['modelo'].nunique()} modelos × 3 semillas × 5 pliegues, por estrato")

## 3. Resultados

In [ ]:
tabla_todos = tabla(res)
tabla_todos

### Por procedencia

Un promedio puede esconder cómo le va a los naturales: el control del Avance 3 mostró que el modelo separa mucho peor los pares que solo tiene NPASS.

In [ ]:
por_procedencia = {e: tabla(res, e) for e in ("solo_chembl", "mixto", "solo_npass")
                   if (res["estrato"] == e).any()}
pd.concat(por_procedencia, names=["procedencia"])[["concordancia", "spearman", "rmse", "mcc", "auroc"]]

### ¿Sobreajusta?

El error en las exactas de entrenamiento contra el de validación. Una brecha grande diría que el modelo memoriza; un modelo lineal con penalización no debería.

In [ ]:
t = tabla_todos[["rmse_entrenamiento", "rmse"]].rename(columns={"rmse": "rmse_validacion"})
t["brecha"] = (t["rmse_validacion"] - t["rmse_entrenamiento"]).round(3)
t

### Qué características pesan

Los coeficientes de la recta, sobre entradas estandarizadas, ajustada con todo entrenamiento y validación. Si la censura cambiara qué importa, los coeficientes de Ridge y Tobit diferirían; se comparan las quince entradas de mayor peso absoluto.

In [ ]:
ridge = baselines()["ridge_exactas"].fit(X, inferior, superior)
tobit = baselines()["tobit"].fit(X, inferior, superior)
coef = pd.DataFrame({"ridge": ridge.modelo[-1].coef_, "tobit": tobit.pesos_}, index=nombres)

top = {k: set(coef[k].abs().nlargest(15).index) for k in coef}
print(f"de las 15 entradas de mayor peso, {len(top['ridge'] & top['tobit'])} coinciden entre Ridge y Tobit")
print(f"correlación de los coeficientes: {coef['ridge'].corr(coef['tobit']):.3f}")
print(f"σ del Tobit (ruido que atribuye a la medición): {tobit.sigma_:.2f} unidades de pActividad")
coef.reindex(coef["tobit"].abs().sort_values(ascending=False).index).head(15).round(3)

### El piso y el techo

El **piso** es `media`. El **techo** es el ruido propio de la medición: repetir el mismo ensayo da diferencias de unas 0.7 unidades logarítmicas en IC50 y 0.5 en Ki (`kalliokoski2013ic50`, `kramer2012ki`; con curación mínima el ruido es mayor, `landrum2024ruido`). Un RMSE muy por debajo de eso no sería un buen resultado sino una señal de fuga.

In [ ]:
print(f"techo de reproducibilidad: {RUIDO_EXPERIMENTAL_IC50} (IC50) y {RUIDO_EXPERIMENTAL_KI} (Ki) unidades logarítmicas")
tabla_todos[["rmse"]].assign(por_encima_del_techo_ic50=lambda t: (t["rmse"] > RUIDO_EXPERIMENTAL_IC50))

## 4. Qué queda establecido, y qué no

(se escribe después de ver los resultados)

In [ ]:
# Registro en experimentos/ (decisión 2.9: la potencia como variable principal).
from src.tracking import Corrida

with Corrida("baseline-regresion", decision="2.9",
             params={"modelos": list(baselines()), "pliegues": 5, "alfa_ridge": ALFA_RIDGE,
                     "objetivo": "pActividad, con las censuradas como cotas (Tobit)",
                     "corte_mcc": 5.0, "estratos": "procedencia"},
             semillas=[0, 1, 2],
             datos=[PROCESADOS / "fase2_pares.parquet",
                    PROCESADOS / "fase2_descriptores_limpios.parquet"]) as c:
    c.metricas({"todos": tabla_todos.to_dict(orient="index"),
                **{e: t.to_dict(orient="index") for e, t in por_procedencia.items()}})

print(c.carpeta.name)